# Notebook 05: Detección y Localización Fina de Daños (Faster R-CNN)

## Contexto y Arquitectura
Implementación y Fine-Tuning de un modelo de Detección de Objetos de dos etapas (*two-stage detector*) para localizar con precisión las autopartes afectadas por el impacto (ópticas, guardabarros, puertas, capó, etc.).

### Decisiones de Diseño Técnico:
- Framework: **PyTorch** / **Torchvision**.
- Arquitectura Base: **Faster R-CNN** con backbone **ResNet-50-FPN** preentrenado.
- Adaptación: Reemplazo del head clasificador (`FastRCNNPredictor`) adaptado al número de clases vehiculares del dataset COCO customizado.
- Métricas: Intersección sobre Unión (IoU) y mAP en cajas delimitadoras (*bounding boxes*).


In [16]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from pycocotools.coco import COCO
from PIL import Image
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.optim import SGD
import os
import config
import matplotlib.pyplot as plt
import matplotlib.patches as patches

### Dataloader Especializado en Formato COCO
Implementamos una clase `Dataset` de PyTorch adaptada para parsear anotaciones de segmentación y cajas delimitadoras (*bounding boxes*) desde los archivos JSON generados para siniestros automotores.

In [4]:
class COCODataset(Dataset):
    def __init__(self, json_file, root_dir, transforms=None):
        self.coco = COCO(json_file)
        self.root_dir = root_dir
        self.transforms = transforms
        self.ids = list(self.coco.imgs.keys())
        
    def __getitem__(self, index):
        image_id = self.ids[index]
        annotations = self.coco.loadAnns(self.coco.getAnnIds(imgIds=image_id))
        image_info = self.coco.loadImgs(image_id)[0]
        image = Image.open(os.path.join(self.root_dir, image_info['file_name'])).convert("RGB")
        
        boxes = []
        labels = []
        for ann in annotations:
            xmin, ymin, width, height = ann['bbox']
            boxes.append([xmin, ymin, xmin + width, ymin + height])
            labels.append(ann['category_id'])
        
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([image_id])
        }
        
        if self.transforms:
            image = self.transforms(image)
        
        return image, target
    
    def __len__(self):
        return len(self.ids)

In [5]:
transform = transforms.Compose([
    transforms.ToTensor()
])

In [6]:
COCO_data_path = config.PROCESSED_PATH_ORIGINAL

In [7]:
train_dataset = COCODataset(
    json_file=F"{COCO_data_path}/json/train.json", 
    root_dir=F"{COCO_data_path}/car-parts/train", 
    transforms=transform)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [8]:
val_dataset = COCODataset(
    json_file=F"{COCO_data_path}/json/test.json", 
    root_dir=F"{COCO_data_path}/car-parts/test", 
    transforms=transform)

val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


### Inicialización y Adaptación de Faster R-CNN
Instanciamos la arquitectura `Faster R-CNN` con Feature Pyramid Network (FPN) y backbone ResNet-50. Reemplazamos el predictor final para que coincida exactamente con las clases de daños y autopartes de nuestro dataset.

In [9]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
num_classes = 6
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

In [10]:
optimizer = SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)

In [11]:
device = torch.device("cpu")

### Bucle de Entrenamiento (Training Loop)
Ejecutamos el ciclo de optimización con descenso de gradiente estocástico (SGD con momentum y weight decay), calculando las pérdidas combinadas de clasificación y regresión de cajas (*bounding box regression*).

In [9]:
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for images, targets in train_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        epoch_loss += losses.item()
    
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': epoch_loss,
    }, F"{config.OBJECT_DETECTION_CHECKPOINT_PATH}/checkpoint-{epoch+1}.pth")
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss}")

Epoch 1/50, Loss: 19.014712318778038
Epoch 2/50, Loss: 11.800283968448639
Epoch 3/50, Loss: 11.564002498984337
Epoch 4/50, Loss: 9.32197830826044
Epoch 5/50, Loss: 7.400040693581104
Epoch 6/50, Loss: 6.418234847486019
Epoch 7/50, Loss: 6.0020706579089165
Epoch 8/50, Loss: 5.0914207100868225
Epoch 9/50, Loss: 4.050155585631728
Epoch 10/50, Loss: 4.446670074015856
Epoch 11/50, Loss: 4.202781718224287
Epoch 12/50, Loss: 3.49122110940516
Epoch 13/50, Loss: 3.3663248270750046
Epoch 14/50, Loss: 3.1524372659623623
Epoch 15/50, Loss: 3.0087761394679546
Epoch 16/50, Loss: 2.617902869358659
Epoch 17/50, Loss: 2.457334265112877
Epoch 18/50, Loss: 2.4737910171970725
Epoch 19/50, Loss: 2.384173220023513
Epoch 20/50, Loss: 2.431715096347034
Epoch 21/50, Loss: 2.3105888850986958
Epoch 22/50, Loss: 2.5180848063901067
Epoch 23/50, Loss: 2.1218462251126766
Epoch 24/50, Loss: 2.1883694995194674
Epoch 25/50, Loss: 2.1820609979331493
Epoch 26/50, Loss: 2.039465021342039
Epoch 27/50, Loss: 1.81710322573781

In [10]:
torch.save(model.state_dict(), config.OBJECT_DETECTION_MODEL_PATH)

### Inferencia y Visualización de Bounding Boxes
Definimos la función de renderizado visual para superponer las predicciones del detector sobre las imágenes de prueba, filtrando detecciones por umbral de confianza (*confidence threshold*).

In [12]:
def plot_detections(image, outputs, threshold=0.5):
    image = image.permute(1, 2, 0).cpu().numpy()
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    
    for i in range(len(outputs['boxes'])):
        score = outputs['scores'][i].item()
        if score > threshold:
            bbox = outputs['boxes'][i].cpu().numpy()
            rect = patches.Rectangle((bbox[0], bbox[1]), bbox[2] - bbox[0], bbox[3] - bbox[1],linewidth=2, edgecolor='r', facecolor='none')
            plt.gca().add_patch(rect)
            plt.text(bbox[0], bbox[1], f"{outputs['labels'][i].item()} {score:.2f}", fontsize=12, color='white', bbox=dict(facecolor='red', alpha=0.5))
    plt.axis('off')
    plt.show()

In [13]:
model.eval()
with torch.no_grad():
    for images, targets in val_loader:
        images = list(image.to(device) for image in images)
        outputs = model(images)

        for img, output in zip(images, outputs):
            plot_detections(img, output)

KeyboardInterrupt: 